# Triadic Cell Notebook v52
## Disagreement-Aware Nexus Gate

v51 found the boundary.

It did **not** clean-lock.

Observed:

$$
\text{base}=0.90625
$$

$$
\text{answerText}=0.916667
$$

$$
\text{blindText}=0.583333
$$

$$
\text{checklist}=0.791667
$$

$$
\text{evidence}=0.875
$$

$$
\text{gated}=0.916667
$$

with:

$$
\text{gated helped}=5,\qquad \text{gated hurt}=4
$$

So the lesson is precise:

$$
\boxed{
\text{blind answer-text is diagnostic, not a driver}
}
$$

and:

$$
\boxed{
\text{answer-text can repair, but it can also overrule a correct base/checklist pair}
}
$$

v52 changes the gate.

Core rule:

$$
\text{do not override a high-margin base answer when checklist agrees with base}
$$

unless there is a stronger two-channel rescue.

The new controller keeps four modes:

1. `base`
2. `answer_text`
3. `checklist`
4. `evidence`

But the **gate** is now disagreement-aware:

- base + checklist agreement blocks answer-text-only overrides,
- answer-text + checklist agreement can rescue base misses,
- answer-text + evidence can rescue if base margin is not locked,
- blind text is recorded but does not drive energy.

Lock target:

$$
\boxed{
\text{gated\_acc} > \text{base\_acc}
\land
\text{gated\_hurt}=0
}
$$


In [ ]:
# Optional installs if needed:
# %pip install -q torch transformers pandas matplotlib tqdm accelerate sentence-transformers

from __future__ import annotations

import os, json, random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    from sentence_transformers import SentenceTransformer
    HAVE_ST = True
except Exception:
    SentenceTransformer = None
    HAVE_ST = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(suppress=True, precision=4)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
print("sentence_transformers:", HAVE_ST)


In [ ]:
# -----------------------------
# CONFIG
# -----------------------------
DATA_JSONL = ""
MAX_SAMPLES = 128

HF_TOKEN = os.getenv("HF_TOKEN", "")
LOCAL_FILES_ONLY = True

MODEL_PROFILE = "qwen25_1p5b_instruct"

MODEL_PROFILES = {
    "qwen25_1p5b_instruct": {
        "MODEL_NAME": "Qwen/Qwen2.5-1.5B-Instruct",
        "EMBED_MODEL_NAME": "sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP": 128,
    },
    "qwen25_3b_instruct": {
        "MODEL_NAME": "Qwen/Qwen2.5-3B-Instruct",
        "EMBED_MODEL_NAME": "sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP": 96,
    },
}

profile = MODEL_PROFILES[MODEL_PROFILE]
MODEL_NAME = profile["MODEL_NAME"]
EMBED_MODEL_NAME = profile["EMBED_MODEL_NAME"]
MAX_SAMPLES = min(MAX_SAMPLES, profile["MAX_SAMPLES_CAP"])

USE_CHAT_TEMPLATE = True
CHOICE_AUDIT_MODE = "rotations"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

# Gate thresholds
BASE_LOCK_MARGIN = 2.0
ANSWER_TEXT_STRONG = 1.0
CHECKLIST_STRONG = 0.65
EVIDENCE_STRONG = 0.55

OUTPUT_DIR = f"v52_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("MODEL:", MODEL_NAME)
print("EMBED:", EMBED_MODEL_NAME)
print("LOCAL_FILES_ONLY:", LOCAL_FILES_ONLY)
print("DEVICE:", DEVICE, "DTYPE:", DTYPE)
print("CHOICE_AUDIT_MODE:", CHOICE_AUDIT_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:
NEXUS_FRAME = """Use the Nexus operational lens.

Rules:
1. Prefer verbs/operations over nouns/labels.
2. Treat shape, constraint, boundary, and gap as primary.
3. A good answer preserves function while hiding complexity inward.
4. For repair questions, start from the needed future state and work backward.
5. Do not choose surface similarity when operational fit is missing.
6. Choose the single best collapse.
"""

NEXUS_CHECKLIST_FRAME = """Evaluate the candidate under the Nexus operational lens.

A correct candidate should satisfy the missing operational shape:
- fits the need,
- preserves or redirects function,
- respects the constraint boundary,
- avoids surface-label traps,
- collapses the missing structure rather than merely naming it.

Score the candidate by operation, not by surface wording.
"""


In [ ]:
ADVERSARIAL_NEXUS_DATA = [
    {"id":"adv_coupler_01","band":"inverse_need_adversarial","prompt":"A spinning rubber coupler is loose on a vacuum pump shaft. The repair must add radial compression while keeping the coupler centered enough to transmit rotation. Which candidate is operationally best?","choices":["tight O-rings seated concentrically around the coupler","a poetic recursive wrap that symbolically surrounds the failure","loose string nearby because string can wrap objects","permanent epoxy locking the coupler off-center"],"answer_idx":0},
    {"id":"adv_coupler_02","band":"inverse_need_adversarial","prompt":"The missing function is not the noun 'rubber part'; it is centered compressive coupling under motion. Which answer preserves that function with the least overbinding?","choices":["a removable radial compression band","a same-named replacement label with no fit data","a clamp that crushes one side harder than the other","a larger motor housing"],"answer_idx":0},
    {"id":"adv_car_01","band":"interface_adversarial","prompt":"A car hides combustion, gearing, sensors, tire friction, steering geometry, and safety constraints. What is the correct interface-collapse?","choices":["a semantic category called vehicle","a readable driver surface: wheel, pedals, seat, motion","a detailed list of engine nouns","a symbol of transportation culture"],"answer_idx":1},
    {"id":"adv_house_01","band":"fold_adversarial","prompt":"A house presents door, room, roof, and shelter. Which answer captures the hidden inward fold rather than the surface noun?","choices":["a building label recognized by zoning language","weather, privacy, load, heat-flow, wiring, plumbing, and human paths folded into shelter","a decorative facade with rooms inside","a static object that stops being computational"],"answer_idx":1},
    {"id":"adv_api_01","band":"interface_adversarial","prompt":"An API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?","choices":["complexity internalized below a stable interface","the implementation stops existing","a name replaces behavior","the public method is only documentation"],"answer_idx":0},
    {"id":"adv_llm_01","band":"ai_runtime_adversarial","prompt":"An LLM answer appears as text, but the output is grown one token at a time. Which candidate fits the runtime?","choices":["a database row copied after lookup","an internal indexed fold-state emits a token and re-indexes","a final paragraph stored whole in a table","a random string independent of previous tokens"],"answer_idx":1},
    {"id":"adv_sha_01","band":"sha_adversarial","prompt":"SHA-256 produces a digest. Under the folding lens, what is the digest?","choices":["randomness created by destroying input structure","a compressed residue of deterministic algebraic folding","semantic meaning extracted from the text","a database pointer to the original message"],"answer_idx":1},
    {"id":"adv_sha_02","band":"sha_adversarial","prompt":"SHA constants and LLM weights are not identical, but their roles rhyme. Which answer states the operational rhyme?","choices":["both are prompts typed by the user","both act as stored structural bias used during folding","both are final answers","both prevent state transitions"],"answer_idx":1},
    {"id":"adv_observable_01","band":"observables_adversarial","prompt":"A recursive loop must load readable information without dissolving into hidden state. What does it need?","choices":["observable residues that can be read inside the loop","only private latent variables with no readout","more nouns in the prompt","a rule forbidding feedback"],"answer_idx":0},
    {"id":"adv_breath_01","band":"observables_adversarial","prompt":"A recursive system breathes without moving matter. What changes?","choices":["the physical object must travel first","resoluteness, tolerance, or admissible-transition pressure","the label attached to the object","nothing can change unless mass moves"],"answer_idx":1},
    {"id":"adv_fold_01","band":"fold_adversarial","prompt":"A folding chair succeeds only if the seated function can return. Which statement captures the fold law?","choices":["the chair becomes smaller by losing its chair function forever","the chair stores deployed geometry inward while preserving recoverable seating","the chair changes category into random metal","the label chair is enough"],"answer_idx":1},
    {"id":"adv_flower_01","band":"fold_adversarial","prompt":"A flower is a visible bloom. Which answer describes the hidden fold rather than surface color?","choices":["pollinator targeting, timing, chemistry, reproduction, symmetry, and genetic memory folded into bloom","only a bright object with petals","a random aesthetic noun","a non-computational decoration"],"answer_idx":0},
    {"id":"adv_tree_01","band":"interface_adversarial","prompt":"A tree exposes leaf, trunk, fruit, and shade. What is hidden below that interface?","choices":["only wood color and branch names","water lift, solar capture, branching optimization, root exchange, seasonal timing, carbon storage","a vehicle-like semantic category","nothing operational"],"answer_idx":1},
    {"id":"adv_surface_01","band":"surface_trap_adversarial","prompt":"A candidate uses the word 'shape' repeatedly but does not fit the socket, preserve function, or respect the boundary. What should the controller do?","choices":["accept it because it contains Nexus vocabulary","reject it because noun/vocabulary match is not operational fit","prefer it because it is longer","ignore the boundary"],"answer_idx":1},
    {"id":"adv_surface_02","band":"surface_trap_adversarial","prompt":"A response gives an impressive theorem name but never shows the fold path, boundary, or preserved function. What is it?","choices":["surface citation without operational collapse","complete proof by label","a physical repair","a valid observable because it sounds formal"],"answer_idx":0},
    {"id":"adv_loose_01","band":"inverse_need_adversarial","prompt":"A temporary field repair must work but release cleanly if the assumption is wrong. Which property matters?","choices":["loose coupling with enough fit to function","maximum permanent binding immediately","semantic agreement with the part name","decorative complexity"],"answer_idx":0},
    {"id":"adv_ping_01","band":"observables_adversarial","prompt":"A ping is a beacon shaped by math. Operationally, what is being tested?","choices":["whether a boundary responds with a matching path","whether a noun label exists in memory","whether the final truth is guaranteed","whether feedback can be avoided"],"answer_idx":0},
    {"id":"adv_socket_01","band":"shape_adversarial","prompt":"A plug works because its prongs meet the socket geometry and allowed transfer. Which relation is primary?","choices":["alphabetic similarity of names","shape-defined permission across a boundary","visual decoration","random contact"],"answer_idx":1},
    {"id":"adv_moore_01","band":"fold_adversarial","prompt":"Moore's law through the folding lens is not just smaller parts. What is the deeper direction?","choices":["more hidden switching complexity per visible unit interface","less complexity everywhere","bigger labels on chips","random miniaturization without function"],"answer_idx":0},
    {"id":"adv_solution_01","band":"solution_adversarial","prompt":"A solution is not merely an answer string. What is it under the Nexus lens?","choices":["the need, constraints, materials, and failure modes folded into the thing that fits","the longest available explanation","a label that resembles the problem","a random future event"],"answer_idx":0},
    {"id":"adv_idea_01","band":"solution_adversarial","prompt":"An idea becomes useful when hidden contradictions and analogies compress into a carryable handle. What is the handle?","choices":["a simple interface over folded cognitive complexity","a decorative sentence only","a noun with no operation","a random memory leak"],"answer_idx":0},
    {"id":"adv_constraint_01","band":"shape_adversarial","prompt":"If shape handles the fold, what is the object doing?","choices":["following the admissible path defined by the constraint field","choosing any collapse path independent of boundary","ignoring the energy basin","proving that constraints are decorative"],"answer_idx":0},
    {"id":"adv_weight_01","band":"ai_runtime_adversarial","prompt":"An LLM weight field is not a lookup table in the database sense. What is it closer to?","choices":["distributed constraint bias shaping the next-token fold","a list of final answers","a file of exact paragraphs","a non-computational object"],"answer_idx":0},
    {"id":"adv_commit_01","band":"solution_adversarial","prompt":"A possible repair is not real until it has potential, a commitment path, and a witness/readout. Which candidate captures that triad?","choices":["stored possibility, realizable transition, observable residue","name, decoration, and confidence","random material, strong opinion, and speed","only the final noun"],"answer_idx":0},
]

def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clean_rows(rows):
    out=[]
    for i,row in enumerate(rows[:MAX_SAMPLES]):
        if not all(k in row for k in ("prompt","choices","answer_idx")):
            print("skip bad row:", row); continue
        if not isinstance(row["choices"], list) or len(row["choices"]) < 2:
            print("skip bad choices:", row); continue
        a=int(row["answer_idx"])
        if not (0 <= a < len(row["choices"])):
            print("skip bad answer_idx:", row); continue
        out.append({
            "id": row.get("id", f"row_{i}"),
            "band": row.get("band", "unknown"),
            "prompt": str(row["prompt"]),
            "choices": [str(x) for x in row["choices"]],
            "answer_idx": a,
        })
    return out

base_rows = clean_rows(load_jsonl(DATA_JSONL) if DATA_JSONL and Path(DATA_JSONL).exists() else ADVERSARIAL_NEXUS_DATA)
print("base rows:", len(base_rows))
pd.DataFrame(base_rows).head()


In [ ]:
def rotate_list(xs, k):
    k = k % len(xs)
    return xs[k:] + xs[:k]

def reorder_row(row, order, suffix):
    old_choices = row["choices"]
    old_answer_text = old_choices[row["answer_idx"]]
    new_choices = [old_choices[i] for i in order]
    new_answer_idx = new_choices.index(old_answer_text)
    return {
        "id": f"{row['id']}__{suffix}",
        "base_id": row["id"],
        "band": row["band"],
        "prompt": row["prompt"],
        "choices": new_choices,
        "answer_idx": new_answer_idx,
        "answer_text": old_answer_text,
        "order": order,
    }

def stable_seed_from_text(text):
    return SEED + sum((i + 1) * ord(ch) for i, ch in enumerate(text)) % 10_000_000

def expand_choice_audit(rows):
    out=[]
    for row in rows:
        n=len(row["choices"])
        if CHOICE_AUDIT_MODE == "none":
            out.append(reorder_row(row, list(range(n)), "orig"))
        elif CHOICE_AUDIT_MODE == "shuffle":
            r=random.Random(stable_seed_from_text(row["id"]))
            order=list(range(n))
            r.shuffle(order)
            out.append(reorder_row(row, order, "shuffle"))
        elif CHOICE_AUDIT_MODE == "rotations":
            for k in range(n):
                order=rotate_list(list(range(n)), k)
                out.append(reorder_row(row, order, f"rot{k}"))
        else:
            raise ValueError(f"Unknown CHOICE_AUDIT_MODE: {CHOICE_AUDIT_MODE}")
    return out

rows = expand_choice_audit(base_rows)
print("expanded rows:", len(rows))
pd.DataFrame(rows)[["id","base_id","band","answer_idx","answer_text","order"]].head(12)


In [ ]:
def hf_kwargs():
    kw = {"local_files_only": LOCAL_FILES_ONLY}
    if HF_TOKEN:
        kw["token"] = HF_TOKEN
    return kw

print("Loading tokenizer/model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **hf_kwargs())
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    **hf_kwargs(),
)
if DEVICE != "cuda":
    lm = lm.to(DEVICE)
lm.eval()
print("Loaded model:", MODEL_NAME)

if EMBED_MODEL_NAME and HAVE_ST:
    print("Using SentenceTransformer embedder")
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
else:
    raise RuntimeError("SentenceTransformer embedder required.")


In [ ]:
LETTERS="ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def maybe_chat(prompt):
    if USE_CHAT_TEMPLATE and hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return prompt

def build_mcq_prompt(row):
    parts=[NEXUS_FRAME, "", "Task:", row["prompt"].strip(), "", "Choices:"]
    for i,c in enumerate(row["choices"]):
        parts.append(f"{LETTERS[i]}. {c}")
    parts += ["", "Return only the single best letter."]
    return "\n".join(parts)

def build_blind_prompt(row):
    return "\n".join([
        NEXUS_FRAME,
        "",
        "Task:",
        row["prompt"].strip(),
        "",
        "No choice list is shown here. Score the candidate by operational fit only.",
        "The Nexus collapse is"
    ])

def conditional_logprob(prefix, suffix):
    prefix_ids=tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids=tokenizer(prefix+suffix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out=lm(full_ids)
        logits=out.logits[:,:-1,:]
        targets=full_ids[:,1:]
        lp=F.log_softmax(logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    start=max(prefix_ids.shape[1]-1, 0)
    return float(lp[:,start:].mean().item())

def normalize_scores(x):
    x=np.array(x,dtype=np.float32)
    return (x-x.mean())/(x.std()+1e-8)

def margin_of(scores):
    order=np.sort(np.array(scores))[::-1]
    return float(order[0]-order[1]) if len(order)>1 else 0.0

def score_base(row):
    p=build_mcq_prompt(row)
    rendered=maybe_chat(p)
    scores=np.array([conditional_logprob(rendered, " "+LETTERS[i]) for i in range(len(row["choices"]))], dtype=np.float32)
    pred=int(np.argmax(scores))
    prob=np.exp(scores-scores.max()); prob=prob/(prob.sum()+1e-8)
    ent=float(-np.sum(prob*np.log(prob+1e-8)))
    return p, scores, prob.astype(np.float32), pred, margin_of(scores), ent

def score_answer_text_with_choices(row, prompt_text):
    prefix = maybe_chat(prompt_text + "\n\nThe Nexus collapse is")
    return np.array([conditional_logprob(prefix, " " + choice) for choice in row["choices"]], dtype=np.float32)

def score_answer_text_blind(row):
    prefix = maybe_chat(build_blind_prompt(row))
    return np.array([conditional_logprob(prefix, " " + choice) for choice in row["choices"]], dtype=np.float32)

def checklist_prompt(row, choice, dimension):
    return "\n".join([
        NEXUS_CHECKLIST_FRAME,
        "",
        "Task:",
        row["prompt"].strip(),
        "",
        f"Candidate: {choice}",
        "",
        f"Checklist dimension: {dimension}",
        "Does this candidate satisfy this dimension?",
        "Answer Fit or Fail."
    ])

CHECKLIST_DIMS = [
    "fits the need",
    "preserves or redirects function",
    "respects the constraint boundary",
    "avoids surface-label traps",
    "collapses the missing structure rather than merely naming it",
]

def score_checklist(row):
    candidate_scores=[]
    dim_scores=[]
    for choice in row["choices"]:
        vals=[]
        for dim in CHECKLIST_DIMS:
            p=maybe_chat(checklist_prompt(row, choice, dim))
            fit=conditional_logprob(p, " Fit")
            fail=conditional_logprob(p, " Fail")
            vals.append(fit-fail)
        dim_scores.append(vals)
        candidate_scores.append(float(np.mean(vals)))
    return np.array(candidate_scores,dtype=np.float32), np.array(dim_scores,dtype=np.float32)

print("Scoring functions loaded.")


In [ ]:
@dataclass
class V52Sample:
    row_id: str
    base_id: str
    band: str
    prompt_text: str
    choices: list
    answer_idx: int
    answer_text: str

    base_scores: np.ndarray
    answer_text_scores: np.ndarray
    blind_text_scores: np.ndarray
    checklist_scores: np.ndarray
    checklist_dim_scores: np.ndarray

    base_pred_idx: int
    base_margin: float
    base_entropy: float

    prompt_vec: np.ndarray
    candidate_vecs: np.ndarray
    pair_vecs: np.ndarray

scored=[]
prompt_texts=[]
choice_texts=[]
pair_texts=[]

for row in tqdm(rows, desc="Scoring base/text/blind/checklist"):
    p,base_scores,base_probs,base_pred,base_margin,base_ent=score_base(row)
    text_scores=score_answer_text_with_choices(row,p)
    blind_scores=score_answer_text_blind(row)
    checklist_scores, checklist_dim_scores = score_checklist(row)

    scored.append((row,p,base_scores,text_scores,blind_scores,checklist_scores,checklist_dim_scores,base_pred,base_margin,base_ent))
    prompt_texts.append(p)
    for i,c in enumerate(row["choices"]):
        choice_texts.append(f"{LETTERS[i]}. {c}")
        pair_texts.append(p+"\n\nCandidate Nexus collapse: "+f"{LETTERS[i]}. {c}")

prompt_embs = embedder.encode(prompt_texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
choice_embs = embedder.encode(choice_texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
pair_embs = embedder.encode(pair_texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)

def l2norm(x, eps=1e-8):
    return x/np.clip(np.linalg.norm(x, axis=-1, keepdims=True), eps, None)

def minmax_local(stack, eps=1e-8):
    mn=stack.min(axis=0, keepdims=True); mx=stack.max(axis=0, keepdims=True)
    return (stack-mn)/np.clip(mx-mn, eps, None)

samples=[]; cp=0; pp=0
for pi,(row,p,base_scores,text_scores,blind_scores,checklist_scores,checklist_dim_scores,base_pred,base_margin,base_ent) in enumerate(scored):
    n=len(row["choices"])
    cand=choice_embs[cp:cp+n]
    pair=pair_embs[pp:pp+n]
    stack=minmax_local(l2norm(np.vstack([prompt_embs[pi:pi+1], cand, pair]).astype(np.float32)))
    samples.append(V52Sample(
        row["id"], row.get("base_id", row["id"]), row["band"], p, row["choices"], row["answer_idx"], row.get("answer_text", row["choices"][row["answer_idx"]]),
        base_scores, text_scores, blind_scores, checklist_scores, checklist_dim_scores,
        base_pred, base_margin, base_ent,
        stack[0], stack[1:1+n], stack[1+n:1+n+n]
    ))
    cp += n
    pp += n

print("prepared:", len(samples))


## v52 Evidence and Gate

Evidence is now driven by:

$$
T_c = \text{answer-text with choices}
$$

plus checklist and base letter score.

Blind text is retained as diagnostic but not in energy.

Energy:

$$
E_i =
-0.85z_{T_c}
-0.45z_{checklist}
-0.20z_{letter}
-0.10z_{pairFit}
+0.30z_{contradiction}
$$

Gate priority:

1. protect base if base and checklist agree strongly,
2. rescue if answer-text and checklist agree,
3. rescue if answer-text and evidence agree and base is not locked,
4. otherwise keep base.

This is a safety gate, not a maximum-score chooser.


In [ ]:
def cos(a,b):
    return float(np.dot(a,b)/((np.linalg.norm(a)+1e-8)*(np.linalg.norm(b)+1e-8)))

def argmax_margin(scores):
    return int(np.argmax(scores)), margin_of(scores)

def folded_evidence(sample: V52Sample):
    k=len(sample.choices)

    letter_z=normalize_scores(sample.base_scores)
    text_z=normalize_scores(sample.answer_text_scores)
    blind_z=normalize_scores(sample.blind_text_scores)
    checklist_z=normalize_scores(sample.checklist_scores)

    pair_fit=np.array([cos(sample.pair_vecs[i], sample.candidate_vecs[i]) for i in range(k)],dtype=np.float32)
    pair_fit_z=normalize_scores(pair_fit)

    contradiction=np.maximum(0.0, -checklist_z) + 0.25*np.abs(text_z-checklist_z)
    contradiction_z=normalize_scores(contradiction)

    energy=(
        -0.85*text_z
        -0.45*checklist_z
        -0.20*letter_z
        -0.10*pair_fit_z
        +0.30*contradiction_z
    )

    pred=int(np.argmin(energy))
    evidence_strength=margin_of(-energy)

    table=pd.DataFrame({
        "choice_idx":list(range(k)),
        "choice":sample.choices,
        "letter_score":sample.base_scores,
        "answer_text_with_choices":sample.answer_text_scores,
        "answer_text_blind":sample.blind_text_scores,
        "checklist_score":sample.checklist_scores,
        "letter_z":letter_z,
        "text_z":text_z,
        "blind_z":blind_z,
        "checklist_z":checklist_z,
        "pair_fit":pair_fit,
        "contradiction_z":contradiction_z,
        "energy":energy,
    })

    return pred, float(evidence_strength), table

def gate_v52(sample, evidence_pred, evidence_margin, text_pred, text_margin, blind_pred, blind_margin, checklist_pred, checklist_margin):
    base_pred=sample.base_pred_idx

    # If evidence equals base, keep base.
    if evidence_pred == base_pred:
        return base_pred, "same_as_base"

    # Hard protection: base is high confidence AND checklist agrees with base.
    # This blocks the v51 adv_coupler_02 failure.
    if base_pred == checklist_pred and sample.base_margin >= BASE_LOCK_MARGIN and checklist_margin >= CHECKLIST_STRONG:
        return base_pred, "protect_base_checklist_agree"

    # Medium protection: checklist agrees with base and answer-text alone is the only dissenter.
    if base_pred == checklist_pred and checklist_margin >= CHECKLIST_STRONG and evidence_pred == text_pred and text_pred != checklist_pred:
        return base_pred, "protect_checklist_against_text_only"

    # Strongest rescue: answer-text and checklist agree against base.
    if evidence_pred == text_pred == checklist_pred and text_margin >= 0.10 and evidence_margin >= 0.05:
        return evidence_pred, "override_text_checklist_evidence_agree"

    # Rescue: answer-text and evidence agree; base is not locked with checklist.
    if evidence_pred == text_pred and text_margin >= ANSWER_TEXT_STRONG and evidence_margin >= 0.10 and sample.base_margin < BASE_LOCK_MARGIN:
        return evidence_pred, "override_text_evidence_base_not_locked"

    # Checklist rescue if evidence agrees and both are strong.
    if evidence_pred == checklist_pred and checklist_margin >= CHECKLIST_STRONG and evidence_margin >= EVIDENCE_STRONG:
        return evidence_pred, "override_checklist_evidence_strong"

    return base_pred, "keep_base_default"

rows_out=[]
evidence_tables={}

for s in samples:
    evidence_pred,evidence_margin,tab=folded_evidence(s)
    evidence_tables[s.row_id]=tab

    text_p,text_margin=argmax_margin(s.answer_text_scores)
    blind_p,blind_margin=argmax_margin(s.blind_text_scores)
    checklist_p,checklist_margin=argmax_margin(s.checklist_scores)

    gated_p,gate_reason=gate_v52(s,evidence_pred,evidence_margin,text_p,text_margin,blind_p,blind_margin,checklist_p,checklist_margin)

    rows_out.append({
        "id":s.row_id,
        "base_id":s.base_id,
        "band":s.band,
        "gold_idx":s.answer_idx,
        "gold_choice":s.choices[s.answer_idx],
        "gold_text":s.answer_text,

        "base_idx":s.base_pred_idx,
        "base_choice":s.choices[s.base_pred_idx],
        "base_correct":int(s.base_pred_idx==s.answer_idx),
        "base_margin":s.base_margin,

        "answer_text_idx":text_p,
        "answer_text_choice":s.choices[text_p],
        "answer_text_correct":int(text_p==s.answer_idx),
        "answer_text_margin":text_margin,

        "blind_text_idx":blind_p,
        "blind_text_choice":s.choices[blind_p],
        "blind_text_correct":int(blind_p==s.answer_idx),
        "blind_text_margin":blind_margin,

        "checklist_idx":checklist_p,
        "checklist_choice":s.choices[checklist_p],
        "checklist_correct":int(checklist_p==s.answer_idx),
        "checklist_margin":checklist_margin,

        "evidence_idx":evidence_pred,
        "evidence_choice":s.choices[evidence_pred],
        "evidence_correct":int(evidence_pred==s.answer_idx),
        "evidence_margin":evidence_margin,

        "gated_idx":gated_p,
        "gated_choice":s.choices[gated_p],
        "gated_correct":int(gated_p==s.answer_idx),
        "gate_reason":gate_reason,
    })

results_df=pd.DataFrame(rows_out)
for mode in ["answer_text","blind_text","checklist","evidence","gated"]:
    results_df[f"{mode}_helped"]=((results_df.base_correct==0)&(results_df[f"{mode}_correct"]==1)).astype(int)
    results_df[f"{mode}_hurt"]=((results_df.base_correct==1)&(results_df[f"{mode}_correct"]==0)).astype(int)

def acc(s): return float(s.mean()) if len(s) else float("nan")

summary=pd.DataFrame([{
    "n":len(results_df),
    "n_base_items":results_df["base_id"].nunique(),
    "choice_audit_mode":CHOICE_AUDIT_MODE,
    "base_acc":acc(results_df.base_correct),
    "answer_text_acc":acc(results_df.answer_text_correct),
    "blind_text_acc":acc(results_df.blind_text_correct),
    "checklist_acc":acc(results_df.checklist_correct),
    "evidence_acc":acc(results_df.evidence_correct),
    "gated_acc":acc(results_df.gated_correct),
    "evidence_gain":acc(results_df.evidence_correct)-acc(results_df.base_correct),
    "gated_gain":acc(results_df.gated_correct)-acc(results_df.base_correct),
    "gated_helped":int(results_df.gated_helped.sum()),
    "gated_hurt":int(results_df.gated_hurt.sum()),
}])

by_band=results_df.groupby("band")[[
    "base_correct","answer_text_correct","blind_text_correct","checklist_correct","evidence_correct","gated_correct",
    "gated_helped","gated_hurt"
]].mean().reset_index()

by_base_item=results_df.groupby(["base_id","band","gold_text"])[[
    "base_correct","answer_text_correct","blind_text_correct","checklist_correct","evidence_correct","gated_correct"
]].mean().reset_index()

interesting=results_df[
    (results_df.base_correct==0)
    | (results_df.evidence_idx != results_df.base_idx)
    | (results_df.gated_idx != results_df.base_idx)
].copy()

display(summary)
display(by_band)
display(by_base_item.sort_values("gated_correct").head(15))
display(interesting[[
    "id","base_id","band","gold_choice",
    "base_choice","base_correct","base_margin",
    "answer_text_choice","answer_text_correct","answer_text_margin",
    "blind_text_choice","blind_text_correct","blind_text_margin",
    "checklist_choice","checklist_correct","checklist_margin",
    "evidence_choice","evidence_correct","evidence_margin",
    "gated_choice","gated_correct","gate_reason"
]].head(40))


In [ ]:
out=Path(OUTPUT_DIR)
results_df.to_csv(out/"results.csv", index=False)
summary.to_csv(out/"summary.csv", index=False)
by_band.to_csv(out/"by_band.csv", index=False)
by_base_item.to_csv(out/"by_base_item.csv", index=False)
interesting.to_csv(out/"interesting_cases.csv", index=False)

for row_id,tab in evidence_tables.items():
    tab.to_csv(out/f"evidence_{row_id}.csv", index=False)

print("Saved outputs in", out)
print("results.csv")
print("summary.csv")
print("by_band.csv")
print("by_base_item.csv")
print("interesting_cases.csv")
print("evidence_*.csv")


In [ ]:
plt.figure(figsize=(11,4))
summary[["base_acc","answer_text_acc","blind_text_acc","checklist_acc","evidence_acc","gated_acc"]].T.plot(kind="bar", legend=False)
plt.title("Accuracy by Scoring Mode")
plt.ylabel("accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(14,5))
by_band.set_index("band")[["base_correct","answer_text_correct","checklist_correct","evidence_correct","gated_correct"]].plot(kind="bar")
plt.title("Accuracy by Adversarial Nexus Band")
plt.ylabel("accuracy")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "gated":[results_df.gated_helped.sum(), results_df.gated_hurt.sum()],
    "evidence":[results_df.evidence_helped.sum(), results_df.evidence_hurt.sum()],
    "answer_text":[results_df.answer_text_helped.sum(), results_df.answer_text_hurt.sum()],
}, index=["helped","hurt"]).plot(kind="bar")
plt.title("Help vs Hurt")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
by_base_item["gated_correct"].hist(bins=10)
plt.title("Per-base-item gated correctness across rotations")
plt.xlabel("mean correctness across rotations")
plt.ylabel("count")
plt.tight_layout()
plt.show()


In [ ]:
for row_id in interesting["id"].head(20):
    print("="*100)
    print("CASE:", row_id)
    display(results_df[results_df.id==row_id][[
        "id","base_id","band","gold_choice","base_choice","base_correct","base_margin",
        "answer_text_choice","answer_text_correct","answer_text_margin",
        "blind_text_choice","blind_text_correct","blind_text_margin",
        "checklist_choice","checklist_correct","checklist_margin",
        "evidence_choice","evidence_correct","evidence_margin",
        "gated_choice","gated_correct","gate_reason"
    ]])
    display(evidence_tables[row_id].sort_values("energy"))


## Readout

v52 is successful if it converts the v51 state:

$$
\text{help}=5,\quad \text{hurt}=4
$$

into:

$$
\text{hurt}=0
$$

while preserving some helped cases.

If it gives:

$$
\text{gated\_acc} > \text{base\_acc}
$$

and:

$$
\text{gated\_hurt}=0
$$

then the controller is usable.

If hurt goes to zero but help also collapses, the safe gate is correct but too conservative.

That still gives the next fold:

$$
\boxed{
\text{learn a gate policy from disagreement patterns}
}
$$
